In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv scikit-learn plotly


# MLFP05 — Exercise 7, Part 4: Adapter Modules (Bridge to M6 LoRA)


**WHAT YOU'LL LEARN:**
After completing this section, you will be able to:
- Understand why full fine-tuning is expensive and risky
- Build bottleneck adapter modules that inject small trainable
layers inside a frozen backbone
- Compare parameter efficiency across methods: from-scratch,
frozen head, adapter, and (preview) LoRA
- Visualise the performance-vs-parameters Pareto frontier
- Apply adapter concepts to a multi-tenant AI platform scenario


**PREREQUISITES: Parts 1-3 (baseline, transfer, data efficiency).**

**ESTIMATED TIME: ~25 min**



In [ ]:
from __future__ import annotations

import asyncio
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision

from shared.mlfp05.ex_7 import (
    BATCH_SIZE,
    EPOCHS,
    N_CLASSES,
    OUTPUT_DIR,
    count_params,
    device,
    init_engines,
    load_cifar10,
    train_model,
)


## THEORY — The Parameter Efficiency Spectrum
Transfer learning has a spectrum of approaches, each trading off
between trainable parameters and model capacity:
FROZEN HEAD (Part 2):
- Update ONLY the final classifier layer (~0.05% of params)
- Fastest training, zero risk of "catastrophic forgetting"
- Limited: the backbone features might not be perfectly suited
FULL FINE-TUNING:
- Update ALL parameters (100% trainable)
- Highest potential accuracy
- Risk: "catastrophic forgetting" — the model unlearns ImageNet
features while overfitting to your small dataset
- Storage: need a full copy of the model per task
ADAPTER MODULES (this section):
- Inject small trainable bottleneck layers INSIDE the frozen backbone
- ~5-10% of params trainable — good balance
- Skip connection: adapter starts as identity, so training begins
from the pre-trained features
- Storage: only save the adapter weights (~1-5 MB) per task
LoRA (Module 6):
- Low-rank decomposition: W + A @ B where A is d x r, B is r x d
- ~1-5% of params trainable — best for LLMs with billions of params
- Same idea as adapters but uses matrix decomposition instead of
bottleneck layers
The key insight: you don't need to update every parameter to adapt a
model. Most of the pre-trained knowledge is useful as-is. You only
need to nudge the model slightly toward your specific task.


In [ ]:
print("\n" + "=" * 70)
print("  PART 4: Adapter Modules (Bridge to M6 LoRA)")
print("=" * 70)


## TASK 1 — Load data and set up engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()


## TASK 2 — Build the BottleneckAdapter module
The adapter is a small trainable module: down-project to a bottleneck
dimension, apply a nonlinearity, then up-project back. The SKIP
CONNECTION is critical: it means the adapter starts as an identity
function (because up-project weights are initialised to zero), so
training begins from the pre-trained features, not random noise.


In [ ]:
class BottleneckAdapter(nn.Module):
    """A simple adapter module: down-project, nonlinearity, up-project.

    The skip connection ensures the adapter starts as an identity function
    (initial up-project weights are zero), so training begins from the
    pre-trained features — not random noise.

    Architecture:
        x -> x + up_project(ReLU(down_project(x)))

    Args:
        dim: Input/output dimension (must match the layer it's inserted into)
        bottleneck: Bottleneck dimension (smaller = fewer params, less capacity)
    """

    def __init__(self, dim: int, bottleneck: int = 64):
        super().__init__()
        # TODO: Create two linear layers and zero-init the up-projection
        # Steps:
        #   1. self.down = nn.Linear(dim, bottleneck)
        #   2. self.up = nn.Linear(bottleneck, dim)
        #   3. nn.init.zeros_(self.up.weight)
        #   4. nn.init.zeros_(self.up.bias)
        # Why zero-init? So the adapter starts as identity (output = input)
        ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement skip connection: x + up(relu(down(x)))
        # Hint: return x + self.up(F.relu(self.down(x)))
        ____


### Checkpoint 1 ─────────────────────────────────────────────────────


In [ ]:
adapter_test = BottleneckAdapter(dim=256, bottleneck=64)
test_input = torch.randn(2, 256)
test_output = adapter_test(test_input)
# At init, the adapter is identity (zero up-projection)
assert torch.allclose(
    test_input, test_output, atol=1e-6
), "Adapter should start as identity function (zero-init up-projection)"
adapter_params = sum(p.numel() for p in adapter_test.parameters())
expected_params = 256 * 64 + 64 + 64 * 256 + 256  # down + bias + up + bias
assert (
    adapter_params == expected_params
), f"Adapter params: {adapter_params} vs expected {expected_params}"
# INTERPRETATION: The zero-init trick is crucial. Without it, the
# adapter would add random noise to the pre-trained features on the
# first forward pass, destroying the valuable ImageNet representations.
# With zero-init, training starts from "no change" and gradually learns
# task-specific adjustments.
print(f"  BottleneckAdapter(256, 64): {adapter_params:,} params, starts as identity")
print("--- Checkpoint 1 passed --- adapter module verified\n")


## TASK 3 — Build adapter-augmented ResNet-18
We insert adapters after ResNet's layer3 and layer4 blocks. The
adapter operates on channel-wise pooled features (spatial average
pooling to get a vector per sample, then adapter, then broadcast
back to spatial dimensions).


In [ ]:
class AdaptedBlock(nn.Module):
    """Wraps a ResNet block with a channel-wise adapter."""

    def __init__(self, block: nn.Module, adapter: BottleneckAdapter):
        super().__init__()
        self.block = block
        self.adapter = adapter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Run block, then apply adapter on channel-wise pooled features
        # Steps:
        #   1. out = self.block(x)
        #   2. b, c, h, w = out.shape
        #   3. pooled = out.mean(dim=[2, 3])  -> (B, C)
        #   4. adapted = self.adapter(pooled)  -> (B, C)
        #   5. return out + adapted.unsqueeze(-1).unsqueeze(-1)
        # Hint: unsqueeze broadcasts (B,C) back to (B,C,1,1) for addition
        ____


def build_adapter_resnet(
    n_classes: int = N_CLASSES,
    bottleneck: int = 64,
) -> nn.Module:
    """ResNet-18 with bottleneck adapters after layer3 and layer4."""
    # TODO: Build adapter-augmented ResNet-18
    # Steps:
    #   1. Load pre-trained ResNet-18 (same as Part 2)
    #   2. Freeze all parameters
    #   3. Replace model.fc with nn.Linear(in_features, n_classes)
    #   4. Wrap model.layer3 with AdaptedBlock(original_layer3, BottleneckAdapter(256, bottleneck))
    #   5. Wrap model.layer4 with AdaptedBlock(original_layer4, BottleneckAdapter(512, bottleneck))
    # Hint: Save original layers before replacing:
    #   original_layer3 = model.layer3
    #   model.layer3 = AdaptedBlock(original_layer3, BottleneckAdapter(256, bottleneck))
    ____


adapter_model = build_adapter_resnet(bottleneck=64)
adapter_model.to(device)
n_adapter_trainable = count_params(adapter_model, trainable_only=True)
n_adapter_total = count_params(adapter_model)

print(
    f"  Adapter ResNet-18: {n_adapter_trainable:,} trainable / "
    f"{n_adapter_total:,} total "
    f"({100 * n_adapter_trainable / n_adapter_total:.1f}%)"
)


### Checkpoint 2 ─────────────────────────────────────────────────────


In [ ]:
assert n_adapter_trainable > 5000, "Adapter should have more params than frozen head"
assert n_adapter_trainable < n_adapter_total, "Should have fewer trainable than total"
# INTERPRETATION: The adapter adds ~100K trainable parameters on top of
# the ~5K frozen-head params. This is still far fewer than the ~11M
# total, but gives the model more capacity to adapt to the task.
print("--- Checkpoint 2 passed --- adapter ResNet built\n")


## TASK 4 — Train and compare all three methods


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: All three approaches on CIFAR-10")
print("=" * 70)

# TODO: Train all three models and collect results
# Method 1: Adapter model (already built above)
# Method 2: Frozen head (build_frozen_head function)
# Method 3: From scratch (build_scratch_cnn function)
# Hint: Use train_model() from helpers for each
# Hint: train_model returns (losses, val_accs, train_accs)

# Method 1: Adapter model
adapter_losses, adapter_accs, _ = train_model(
    adapter_model,
    "adapter_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_adapter = max(adapter_accs)


# Method 2: Frozen head (from Part 2)
def build_frozen_head(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Same as Part 2 — load ResNet-18, freeze, replace fc
    ____


frozen_model = build_frozen_head()
frozen_losses, frozen_accs, _ = train_model(
    frozen_model,
    "frozen_head_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_frozen = max(frozen_accs)
n_frozen_trainable = count_params(frozen_model, trainable_only=True)


# Method 3: From scratch
def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Same 3-layer CNN from Part 1
    ____


scratch_model = build_scratch_cnn()
scratch_losses, scratch_accs, _ = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_scratch = max(scratch_accs)
n_scratch_trainable = count_params(scratch_model, trainable_only=True)


### Checkpoint 3 ─────────────────────────────────────────────────────


In [ ]:
assert (
    best_adapter > 0.20
), f"Adapter model should beat random chance (acc={best_adapter:.3f})"
assert (
    n_adapter_trainable < n_adapter_total
), "Adapter model should have fewer trainable params than total"
print(f"\n  === Parameter Efficiency Comparison ===")
print(f"  {'Method':<25} {'Trainable':>12} {'Val Acc':>10}")
print("  " + "-" * 50)
print(f"  {'From scratch':<25} {n_scratch_trainable:>12,} {best_scratch:>10.4f}")
print(f"  {'Frozen head':<25} {n_frozen_trainable:>12,} {best_frozen:>10.4f}")
print(
    f"  {'Adapter (bottleneck)':<25} {n_adapter_trainable:>12,} {best_adapter:>10.4f}"
)
print("\n--- Checkpoint 3 passed --- all three methods compared\n")


## TASK 5 — Visualise: Parameter count vs performance Pareto chart


In [ ]:
# TODO: Create two visualisations:
# 1. Pareto chart: trainable params (x, log scale) vs accuracy (y)
#    - Scatter with 3 methods + LoRA preview point
#    - LoRA estimate: ~2% of total params, ~98% of adapter accuracy
# 2. Training curves: epoch vs accuracy for all three methods
# Hint: fig_pareto = go.Figure()
# Hint: fig_pareto.add_trace(go.Scatter(x=param_counts, y=accuracies, mode="markers+text"))
# Hint: fig_pareto.update_layout(xaxis_type="log")

methods = ["From Scratch", "Frozen Head", "Adapter (bottleneck=64)"]
param_counts = [n_scratch_trainable, n_frozen_trainable, n_adapter_trainable]
accuracies = [best_scratch, best_frozen, best_adapter]
colours = ["#FF5722", "#2196F3", "#4CAF50"]

fig_pareto = go.Figure()
____

pareto_path = OUTPUT_DIR / "04_parameter_pareto.html"
fig_pareto.write_html(str(pareto_path))
print(f"  Saved: {pareto_path}")

# Training curves comparison
fig_curves = go.Figure()
epochs_x = list(range(1, EPOCHS + 1))
____

curves_path = OUTPUT_DIR / "04_training_curves.html"
fig_curves.write_html(str(curves_path))
print(f"  Saved: {curves_path}")


### Checkpoint 4 ─────────────────────────────────────────────────────


In [ ]:
assert pareto_path.exists(), "Pareto chart should be saved"
assert curves_path.exists(), "Training curves should be saved"
print("--- Checkpoint 4 passed --- visualisations complete\n")


## TASK 6 — Apply: Multi-Tenant AI Platform in Singapore
SCENARIO: You're building an AI platform in Singapore that serves
multiple clients. Each client needs a custom image classifier, but
they share the same base architecture (ResNet-18).
Full fine-tuning: store 11M params per client = ~44 MB per model
Adapter approach: store ~100K params per client = ~0.4 MB per adapter
For 50 clients, that's 2.2 GB vs 20 MB. And at inference time, you
can keep ONE ResNet-18 in GPU memory and swap adapters per request.


In [ ]:
print("\n" + "=" * 70)
print("  APPLY: Multi-Tenant AI Platform — One Base, Many Adapters")
print("=" * 70)

N_CLIENTS = 50
FULL_MODEL_MB = n_adapter_total * 4 / (1024 * 1024)  # float32, 4 bytes each
ADAPTER_MB = n_adapter_trainable * 4 / (1024 * 1024)

# TODO: Print the multi-tenant storage analysis
# Steps:
#   1. Print per-client and 50-client storage for full fine-tuning
#   2. Print per-client and 50-client storage for adapter approach
#      (adapter: 1 base model + N tiny adapters)
#   3. Calculate savings_storage and savings_gpu
#   4. Print inference cost comparison
# Hint: Full = N_CLIENTS * FULL_MODEL_MB
# Hint: Adapter = FULL_MODEL_MB + N_CLIENTS * ADAPTER_MB (one shared base)
# Hint: GPU at inference: full needs N models, adapter needs 1 base + swap
print(f"\n  === Multi-Tenant Storage Analysis (50 clients) ===")
print(f"  Base model (ResNet-18): {n_adapter_total:,} params = {FULL_MODEL_MB:.1f} MB")
print(f"  Adapter weights: {n_adapter_trainable:,} params = {ADAPTER_MB:.2f} MB")
____

savings_storage = N_CLIENTS * FULL_MODEL_MB - (FULL_MODEL_MB + N_CLIENTS * ADAPTER_MB)
savings_gpu = N_CLIENTS * FULL_MODEL_MB - (FULL_MODEL_MB + ADAPTER_MB)

print(
    f"\n  Storage savings: {savings_storage:.0f} MB ({savings_storage / (N_CLIENTS * FULL_MODEL_MB) * 100:.0f}%)"
)
print(f"  GPU memory savings: {savings_gpu:.0f} MB (load one base + swap adapters)")
print()
print(f"  BRIDGE TO M6:")
print(f"  In M6, you will learn LoRA (Low-Rank Adaptation) — the adapter")
print(f"  technique designed for LLMs with BILLIONS of parameters.")
print(f"  Same concept: inject small trainable matrices into frozen weights.")
print(f"  LoRA uses low-rank decomposition: W + A @ B where rank r << d.")
print(f"  A GPT-scale model with 7B params might need only ~4M LoRA params (0.06%).")


### Checkpoint 5 ─────────────────────────────────────────────────────


In [ ]:
assert savings_storage > 0, "Adapter approach should save storage"
assert ADAPTER_MB < FULL_MODEL_MB, "Adapter should be smaller than full model"
# INTERPRETATION: Adapters make multi-tenant ML serving economically
# viable. Instead of N copies of the full model (one per client), you
# store one shared backbone plus N tiny adapter files. This is the
# same principle that makes LoRA practical for LLM fine-tuning in M6.
print("\n--- Checkpoint 5 passed --- multi-tenant analysis complete\n")


## CLEANUP


In [ ]:
await conn.close()


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  PART 4 COMPLETE — What You've Learned")
print("=" * 70)
print(
    f"""
  [x] Built a BottleneckAdapter module with zero-init skip connection
  [x] Inserted adapters into frozen ResNet-18 (layer3 + layer4)
  [x] Compared three approaches:
      From scratch: {n_scratch_trainable:>8,} params -> {best_scratch:.1%} accuracy
      Frozen head:  {n_frozen_trainable:>8,} params -> {best_frozen:.1%} accuracy
      Adapter:      {n_adapter_trainable:>8,} params -> {best_adapter:.1%} accuracy
  [x] Visualised the parameter-efficiency Pareto frontier
  [x] Analysed multi-tenant serving: {savings_storage:.0f} MB saved across 50 clients

  THE TRANSFER LEARNING SPECTRUM:
    Frozen head  ->  Adapter/LoRA  ->  Partial fine-tune  ->  Full fine-tune
    (fewest params)                                         (most params)
    (fastest training)                                      (highest capacity)
    (safest from forgetting)                                (risk of forgetting)

  BRIDGE TO M6 — LoRA:
    Adapters: bottleneck layers (down -> ReLU -> up)
    LoRA:     low-rank matrices (W + A @ B, where r << d)
    Same principle, different math. LoRA is preferred for LLMs because
    the low-rank decomposition is more parameter-efficient at scale.

  NEXT: Part 5 deploys the best model to production with ONNX export
  and InferenceServer — the full pipeline from experiment to serving.
"""
)
